<a href="https://colab.research.google.com/github/vappanna/BerAIML/blob/main/MachineDemandPlanning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error
from google.colab import files
import io

def run_machine_sales_forecast_with_model_column():
    print("--- STEP 1: DATA UPLOAD ---")
    uploaded = files.upload()

    if not uploaded:
        print("❌ No file uploaded.")
        return

    file_name = list(uploaded.keys())[0]
    df = pd.read_csv(io.BytesIO(uploaded[file_name]))
    df = df.sort_values(['Region', 'Machine', 'Quarter'])

    future_quarters = ['2026Q3', '2026Q4', '2027Q1', '2027Q2', '2027Q3', '2027Q4']
    forecast_list = []
    performance_metrics = []

    print("\n--- STEP 2: RUNNING MODEL TOURNAMENT ---")
    for (region, machine), group in df.groupby(['Region', 'Machine']):
        series = group['Install_Count'].values
        if len(series) < 4:
            continue

        try:
            # Backtest (Validation on last 2 quarters)
            train, test = series[:-2], series[-2:]

            # 1. Holt-Winters
            hw_model = ExponentialSmoothing(train, trend='add').fit()
            hw_preds = hw_model.forecast(2)
            hw_rmse = np.sqrt(mean_squared_error(test, hw_preds))

            # 2. Moving Average (Fallback)
            ma_preds = [np.mean(train[-4:])] * 2
            ma_rmse = np.sqrt(mean_squared_error(test, ma_preds))

            # WINNER SELECTION
            if hw_rmse <= ma_rmse:
                final_model = ExponentialSmoothing(series, trend='add').fit()
                future_preds = final_model.forecast(6)
                winner = "Holt-Winters"
                win_rmse = hw_rmse
            else:
                future_preds = [np.mean(series[-4:])] * 6
                winner = "Moving Average"
                win_rmse = ma_rmse

            # Calculate Accuracy Percentages
            avg_val = np.mean(test) if np.mean(test) > 0 else 1
            mape = (win_rmse / avg_val) * 100

            # THE FIX: Ensuring 'Model_Used' is explicitly added here
            performance_metrics.append({
                'Region': region,
                'Machine': machine,
                'Model_Used': winner,  # <--- It's back!
                'RMSE': round(win_rmse, 2),
                'RMSE_Pct': f"{round(mape, 2)}%",
                'MAE': round(mean_absolute_error(test, (hw_preds if winner=="Holt-Winters" else ma_preds)), 2)
            })

            for q, p in zip(future_quarters, future_preds):
                forecast_list.append({
                    'Region': region, 'Machine': machine, 'Quarter': q,
                    'Projected_Install_Count': max(0, round(p, 0))
                })
        except:
            continue

    # CONSOLIDATE
    df_forecast = pd.DataFrame(forecast_list)
    df_metrics = pd.DataFrame(performance_metrics)

    # DOWNLOADS
    df_forecast.to_csv("Machine_Sales_Projection_2027.csv", index=False)
    df_metrics.to_csv("Machine_Model_Metrics_Final.csv", index=False)

    print("\n✅ DONE! Check 'Machine_Model_Metrics_Final.csv' for the Model_Used column.")
    files.download("Machine_Sales_Projection_2027.csv")
    files.download("Machine_Model_Metrics_Final.csv")

# EXECUTE
run_machine_sales_forecast_with_model_column()

--- STEP 1: DATA UPLOAD ---


Saving Historical__and_future.csv to Historical__and_future (4).csv

--- STEP 2: RUNNING MODEL TOURNAMENT ---


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/holtwinters/model.py:1380: RuntimeWarning: divide by zero encountered in log
  aic = self.nobs * np.log(sse / self.nobs) + k * 2
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/holtwinters/model.py:1386: RuntimeWarning: invalid value encountered in scalar add
  aicc = aic + aicc_penalty
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/holtwinters/model.py:1387: RuntimeWarning: divide by zero encountered in log
  bic = self.nobs * np.log(sse / self.nobs) + k * np.log(self.nobs)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/holtwinters/model.py:1380: RuntimeWarning: divide by zero encountered in log
  aic = self.nobs * np.log(sse / self.nobs) + k * 2
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/holtwinters/model.py:1386: RuntimeWarning: invalid value encountered in scalar add
  aicc = aic + aicc_penalty
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/holtwinters/model.py:1387: RuntimeWarni


✅ DONE! Check 'Machine_Model_Metrics_Final.csv' for the Model_Used column.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>